# Pipeline Completa — Modello SIR come Catena di Markov

**Corso**: Modelli Probabilistici — Università di Bologna  
**Docente**: Prof. Salvatore Federico  
**Studente**: Francesco Castaldi  
**A.A.**: 2025/2026

---

Questo notebook percorre l'intera pipeline del progetto:
dalla definizione matematica della catena fino all'analisi di sensitività.

| Sezione | Contenuto |
|---------|----------|
| 1 | Setup e import dei moduli |
| 2 | Parametri del modello e $R_0$ |
| 3 | Un passo della catena: `next_state()` |
| 4 | Matrice di transizione esplicita (heatmap) |
| 5 | Singola traiettoria stocastica |
| 6 | Simulazione Monte Carlo (M repliche) |
| 7 | Statistiche descrittive: picco, $\tau$, $R_\infty$ |
| 8 | Confronto con ODE deterministica |
| 9 | Analisi di sensitività ($\beta$, $\gamma$ variabili) |
| 10 | Riepilogo e interpretazione |

---
## 1. Setup e Import

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, "..")

from src.model import next_state, transition_matrix, N, BETA, GAMMA, I0, T_MAX, M
from src.simulation import run_single, run_montecarlo
from src.analysis import extinction_time, compute_stats, solve_ode_sir, get_mc_mean_std
from src.plotting import pad_results

print("OK — Tutti i moduli importati.")

---
## 2. Parametri del Modello

Il modello SIR è descritto da sei parametri:

| Simbolo | Valore | Significato |
|---------|--------|-------------|
| $N$ | 100 | Popolazione totale |
| $\beta$ | 0.2 | Probabilità di contagio per contatto |
| $\gamma$ | 0.1 | Probabilità di guarigione |
| $I_0$ | 5 | Infetti iniziali |
| $T_{\max}$ | 200 | Orizzonte temporale massimo |
| $M$ | 1000 | Numero di simulazioni Monte Carlo |

**Numero di riproduzione di base**:

$$R_0 = \frac{\beta}{\gamma} = \frac{0.2}{0.1} = 2.0$$

Interpretazione: ogni infetto genera in media 2 nuovi infetti. Poiché $R_0 > 1$, l'epidemia si propaga.

In [ ]:
R0 = BETA / GAMMA
print(f"Parametri del modello:")
print(f"  N = {N}")
print(f"  beta = {BETA}")
print(f"  gamma = {GAMMA}")
print(f"  I0 = {I0}")
print(f"  T_MAX = {T_MAX}")
print(f"  M = {M}")
print(f"  R0 = beta/gamma = {BETA}/{GAMMA} = {R0}")

---
## 3. Un Passo della Catena: `next_state()`

Il nucleo della catena di Markov è la funzione `next_state(s, i, r)`, che:
1. Se $I = 0$: lo stato è **assorbente** → restituisce $(s, 0, r)$
2. Altrimenti:
   - **Contagi**: $C \sim \text{Binomial}(s,\, \beta \cdot i / n)$
   - **Guarigioni**: $G \sim \text{Binomial}(i,\, \gamma)$
   - Nuovo stato: $s' = s - C,\; i' = i + C - G,\; r' = r + G$

La probabilità di transizione da $(s,i,r)$ a $(s',i',r')$ è:

$$P_{(s,i,r)\to(s',i',r')} = \binom{s}{c}(\beta i/n)^c(1-\beta i/n)^{s-c} \cdot \binom{i}{g}\gamma^g(1-\gamma)^{i-g}$$

In [ ]:
# Esempio: partiamo da uno stato intermedio
s0, i0, r0 = 80, 10, 10
print(f"Stato iniziale: S={s0}, I={i0}, R={r0}")

for t in range(5):
    s0, i0, r0 = next_state(s0, i0, r0, N, BETA, GAMMA)
    print(f"  Passo {t+1}: S={s0}, I={i0}, R={r0}")
    if i0 == 0:
        print(f"  -> Assorbimento raggiunto al passo {t+1}")
        break

---
## 4. Matrice di Transizione Esplicita

Per popolazione piccola ($N \leq 5$) possiamo calcolare esplicitamente la matrice di transizione $P$.
La matrice ha dimensione $n_{\text{stati}} \times n_{\text{stati}}$ dove gli stati sono tutte le triple $(s,i,r)$ con $s+i+r=N$.

Per $N=4$ ci sono 15 stati. La heatmap mostra la struttura: le righe con $I=0$ hanno probabilità 1 sulla diagonale (stati assorbenti).

In [ ]:
# Calcola matrice di transizione per N=4
n_small = 4
P, states = transition_matrix(n_small, beta=0.5, gamma=0.3)
n_states = len(states)

print(f"N={n_small}: {n_states} stati possibili")
print(f"Matrice P shape: {P.shape}")
print(f"Somma riga 0 (stocastica): {P[0].sum():.6f}")
print()
print("Stati (s,i,r):")
for idx, st in enumerate(states):
    assorbente = " (ASSORBENTE)" if st[1] == 0 else ""
    print(f"  {idx}: {st}{assorbente}")

In [ ]:
# Heatmap della matrice di transizione
from src.plotting import plot_transition_heatmap

fig = plt.figure(figsize=(10, 8))
plt.imshow(P, cmap="YlOrRd", vmin=0, vmax=1, aspect="equal")

state_labels = [f"({s},{i},{r})" for (s,i,r) in states]
for i in range(n_states):
    for j in range(n_states):
        val = P[i, j]
        if val > 0.005:
            plt.text(j, i, f"{val:.2f}", ha="center", va="center",
                     fontsize=7, color="white" if val > 0.6 else "black")

plt.xticks(range(n_states), state_labels, rotation=90, fontsize=7)
plt.yticks(range(n_states), state_labels, fontsize=7)
plt.xlabel("Stato di arrivo")
plt.ylabel("Stato di partenza")
plt.title(f"Matrice di transizione P — N={n_small}, beta=0.5, gamma=0.3")
plt.colorbar(shrink=0.8, label="Probabilita di transizione")
plt.tight_layout()
plt.show()

print("Le righe con I=0 hanno P[i,i]=1 (stati assorbenti).")
print(f"Verifica stocasticita: righe sommano a 1? {np.allclose(P.sum(axis=1), 1.0)}")

---
## 5. Singola Traiettoria Stocastica

Simuliamo una singola evoluzione del modello SIR. La traiettoria termina quando $I=0$ (estinzione) o quando si raggiunge $T_{\max}$.

In [ ]:
np.random.seed(42)
traj = run_single()
tau = extinction_time(traj)

plt.figure(figsize=(10, 4))
steps = range(len(traj))
plt.plot(steps, traj[:, 0], label="S (Suscettibili)", color="steelblue", linewidth=1.5)
plt.plot(steps, traj[:, 1], label="I (Infetti)", color="firebrick", linewidth=1.5)
plt.plot(steps, traj[:, 2], label="R (Rimossi)", color="seagreen", linewidth=1.5)
plt.axvline(tau, color="gray", linestyle=":", alpha=0.7, label=f"Estinzione tau = {tau}")
plt.xlabel("Passo temporale t")
plt.ylabel("Individui")
plt.title(f"Singola traiettoria SIR (beta={BETA}, gamma={GAMMA})")
plt.legend()
plt.tight_layout()
plt.show()

print(f"Tempo di estinzione: tau = {tau}")
print(f"Picco infetti: I_max = {int(traj[:, 1].max())}")
print(f"Rimossi finali: R_inf = {int(traj[-1, 2])}")
print(f"Conservazione N: S+ I + R = {int(traj[-1, 0] + traj[-1, 1] + traj[-1, 2])} (atteso {N})")

---
## 6. Simulazione Monte Carlo

Eseguiamo $M$ repliche indipendenti per studiare la distribuzione degli esiti.
Ogni traiettoria ha lunghezza diversa (dipende dal tempo di estinzione), quindi usiamo `pad_results()` per allinearle.

In [ ]:
np.random.seed(42)
results = run_montecarlo(m=1000)
print(f"Eseguite {len(results)} simulazioni Monte Carlo.")

# Allinea le traiettorie
s_mat, i_mat, r_mat, max_len = pad_results(results)
print(f"Lunghezza massima traiettorie: {max_len}")
print(f"Matrice infetti shape: {i_mat.shape} (M x max_len, con NaN dove mancano dati)")

In [ ]:
# Traiettoria media +/- 1 std
steps = range(max_len)

plt.figure(figsize=(10, 4))
for mat, color, label in [
    (s_mat, "steelblue", "S"),
    (i_mat, "firebrick", "I"),
    (r_mat, "seagreen", "R"),
]:
    mean = np.nanmean(mat, axis=0)
    std = np.nanstd(mat, axis=0)
    plt.plot(steps, mean, color=color, label=f"E[{label}]")
    plt.fill_between(steps, mean - std, mean + std, color=color, alpha=0.15)

plt.xlabel("Passo temporale t")
plt.ylabel("Individui")
plt.title(f"Traiettorie medie +/- 1 std — {len(results)} simulazioni MC")
plt.legend()
plt.tight_layout()
plt.show()

---
## 7. Statistiche Descrittive

Calcoliamo tre indicatori chiave su tutte le simulazioni:
- **Picco infetti** $I_{\max}$: massimo numero di infetti simultanei
- **Tempo di estinzione** $\tau$: primo istante in cui $I=0$
- **Rimossi finali** $R_\infty$: numero di guariti a fine epidemia

In [ ]:
stats = compute_stats(results)

print(f"=== Statistiche su {len(results)} simulazioni ===")
print(f"  Picco infetti I_max:    {stats['mean_peak_infected']:.1f} ± {stats['std_peak_infected']:.1f}")
print(f"  Tempo estinzione tau:   {stats['mean_extinction_time']:.1f} ± {stats['std_extinction_time']:.1f}")
print(f"  Massimo tau osservato:  {stats['max_extinction_time']:.0f}")
print(f"  Rimossi finali R_inf:   {stats['mean_r_inf']:.1f} ± {stats['std_r_inf']:.1f}")
print(f"  R0 = beta/gamma =       {BETA/GAMMA:.2f}")

In [ ]:
# Istogramma del tempo di estinzione
taus = [extinction_time(r) for r in results]
peaks = [float(np.max(r[:, 1])) for r in results]
r_infs = [float(r[-1, 2]) for r in results]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(peaks, bins=25, color="firebrick", edgecolor="white", alpha=0.8)
axes[0].axvline(np.mean(peaks), color="black", linestyle="--",
                label=f"Media = {np.mean(peaks):.1f}")
axes[0].set_xlabel("Picco infetti I_max")
axes[0].set_ylabel("Frequenza")
axes[0].set_title("Distribuzione del picco")
axes[0].legend()

axes[1].hist(taus, bins=25, color="slateblue", edgecolor="white", alpha=0.8)
axes[1].axvline(np.mean(taus), color="black", linestyle="--",
                label=f"Media = {np.mean(taus):.1f}")
axes[1].set_xlabel("Tempo di estinzione tau")
axes[1].set_ylabel("Frequenza")
axes[1].set_title("Distribuzione di tau")
axes[1].legend()

axes[2].hist(r_infs, bins=25, color="seagreen", edgecolor="white", alpha=0.8)
axes[2].axvline(np.mean(r_infs), color="black", linestyle="--",
                label=f"Media = {np.mean(r_infs):.1f}")
axes[2].set_xlabel("Rimossi finali R_inf")
axes[2].set_ylabel("Frequenza")
axes[2].set_title("Distribuzione di R_inf")
axes[2].legend()

plt.tight_layout()
plt.show()

---
## 8. Confronto con ODE Deterministica

Il modello SIR può essere descritto anche da un sistema di equazioni differenziali ordinarie (ODE):

$$\frac{dS}{dt} = -\beta \frac{SI}{N}, \quad \frac{dI}{dt} = \beta \frac{SI}{N} - \gamma I, \quad \frac{dR}{dt} = \gamma I$$

Risolviamo numericamente con Eulero esplicito e confrontiamo con la media Monte Carlo.
La soluzione ODE rappresenta il **limite deterministico** per $N \to \infty$.

In [ ]:
# Soluzione ODE
t_ode, s_ode, i_ode, r_ode = solve_ode_sir(N, BETA, GAMMA, I0, T_MAX)

# Media MC
mc = get_mc_mean_std(results)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Pannello 1: Infetti MC vs ODE
axes[0].plot(mc["t"], mc["i_mean"], color="firebrick", label="MC media E[I]")
axes[0].fill_between(mc["t"],
                     np.maximum(mc["i_mean"] - mc["i_std"], 0),
                     mc["i_mean"] + mc["i_std"],
                     color="firebrick", alpha=0.12, label="MC +/- 1 std")
axes[0].plot(t_ode, i_ode, color="black", linestyle="--", linewidth=2,
             label="ODE deterministica")
axes[0].set_xlabel("Passo temporale t")
axes[0].set_ylabel("Infetti I")
axes[0].set_title("Confronto: MC vs ODE — Infetti")
axes[0].legend()

# Pannello 2: Tutti i compartimenti ODE
axes[1].plot(t_ode, s_ode, color="steelblue", label="S (ODE)")
axes[1].plot(t_ode, i_ode, color="firebrick", label="I (ODE)")
axes[1].plot(t_ode, r_ode, color="seagreen", label="R (ODE)")
axes[1].scatter(mc["t"][::5], mc["s_mean"][::5], color="steelblue", s=8, alpha=0.4, label="S (MC)")
axes[1].scatter(mc["t"][::5], mc["i_mean"][::5], color="firebrick", s=8, alpha=0.4, label="I (MC)")
axes[1].scatter(mc["t"][::5], mc["r_mean"][::5], color="seagreen", s=8, alpha=0.4, label="R (MC)")
axes[1].set_xlabel("Passo temporale t")
axes[1].set_ylabel("Individui")
axes[1].set_title("Confronto completo S/I/R — ODE (linee) vs MC (punti)")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

print(f"Picco ODE: I_max = {i_ode.max():.1f}")
print(f"Picco MC:  I_max = {mc['i_mean'].max():.1f} +/- {mc['i_std'].max():.1f}")
print(f"R_inf ODE: {r_ode[-1]:.1f}")
print(f"R_inf MC:  {mc['r_mean'][-1]:.1f} +/- {mc['r_std'][-1]:.1f}")

---
## 9. Analisi di Sensitività

Variamo $\beta$ e $\gamma$ per osservare l'effetto su $I_{\max}$, $\tau$ e $R_\infty$.
Per ogni scenario calcoliamo $R_0 = \beta/\gamma$.

In [ ]:
scenari = [
    ("beta=0.15, gamma=0.1", 0.15, 0.1, "steelblue"),
    ("beta=0.20, gamma=0.1", 0.20, 0.1, "firebrick"),
    ("beta=0.25, gamma=0.1", 0.25, 0.1, "seagreen"),
    ("beta=0.20, gamma=0.05", 0.20, 0.05, "darkorange"),
    ("beta=0.20, gamma=0.15", 0.20, 0.15, "purple"),
]

np.random.seed(42)

print(f"{'Scenario':<22} {'R0':>5} {'Picco I':>8} {'tau medio':>8} {'Rinf medio':>8}")
print("-" * 55)

plt.figure(figsize=(10, 4))

for label, beta, gamma, color in scenari:
    r0 = beta / gamma
    res = run_montecarlo(m=500, n=N, i0=I0, t_max=T_MAX, beta=beta, gamma=gamma)
    s = compute_stats(res)
    
    # Media infetti per il plot
    s_m, i_m, r_m, ml = pad_results(res)
    mean_i = np.nanmean(i_m, axis=0)
    plt.plot(range(len(mean_i)), mean_i, color=color, label=label, alpha=0.8)
    
    print(f"{label:<22} {r0:>5.2f} {s['mean_peak_infected']:>8.1f} "
          f"{s['mean_extinction_time']:>8.1f} {s['mean_r_inf']:>8.1f}")

plt.xlabel("Passo temporale t")
plt.ylabel("Infetti medi E[I]")
plt.title("Confronto scenari: effetto di beta e gamma")
plt.legend()
plt.tight_layout()
plt.show()

print()
print("Osservazioni:")
print("- Aumentando beta (trasmissione), picco e R_inf crescono")
print("- Aumentando gamma (guarigione), epidemia si attenua")
print(f"- R0 > 1 necessario per propagazione epidemica")

---
## 10. Riepilogo

### Catena di Markov SIR — Risultati chiave

| Concetto | Risultato |
|----------|-----------|
| **Modello** | Catena di Markov a tempo discreto su $N+1$ stati |
| **Matrice $P$** | Stocastica, righe con $I=0$ sono assorbenti |
| **Singola traiettoria** | Comportamento stocastico, estinzione garantita in tempo finito |
| **Monte Carlo** | $M=1000$ simulazioni: $I_{\max} \approx 29 \pm 9$, $\tau \approx 68 \pm 18$ |
| **ODE** | Limite deterministico: picco $\approx 18$, $R_\infty \approx 82$ |
| **R0** | $R_0 = 2.0$ → epidemia propagata |
| **Sensitività** | $\beta$ e $\gamma$ controllano picco e durata |

### Interpretazione

Il modello SIR come catena di Markov cattura la **natura stocastica** di un'epidemia
in una popolazione piccola ($N=100$). La variabilità è significativa:
non tutte le simulazioni hanno lo stesso picco o la stessa durata.
La soluzione ODE fornisce il **comportamento medio atteso** per $N \to \infty$,
mentre la simulazione Monte Carlo mostra la **distribuzione completa** degli esiti,
informazione che l'ODE perde.

Gli stati assorbenti ($I=0$) garantiscono che ogni traiettoria termina
con l'estinzione dell'infezione in tempo finito con probabilità 1.

---
*Notebook generato per l'esame orale di Modelli Probabilistici — Prof. Salvatore Federico.*
*Pipeline completa del progetto SIR Markov Chain.*